## **1. Installer et importer les bibliothèques nécessaires**

In [ ]:
# À exécuter seulement si une bibliothèque manque dans ton environnement.
# %pip install pandas numpy matplotlib scikit-learn torch


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset, DataLoader

# Fixer les graines rend les résultats plus reproductibles d'une exécution à l'autre.
np.random.seed(42)
torch.manual_seed(42)


## **2. Charger et corriger le dataset**

In [ ]:
# Charger le fichier CSV.
# Le notebook peut être lancé depuis son dossier ou depuis la racine du projet.
csv_path = Path('YRCW.csv')
if not csv_path.exists():
    csv_path = Path('Week5/day4/DailyChallenge/YRCW.csv')

if not csv_path.exists():
    raise FileNotFoundError(
        "YRCW.csv est introuvable. Place le fichier dans Week5/day4/DailyChallenge avant d'exécuter le notebook."
    )

df = pd.read_csv(csv_path)


In [ ]:
df.head()

In [ ]:
df.info()

### Structure du dataset

Chaque ligne représente une journée de trading.

* `Date` : date de trading
* `Open` : prix d'ouverture
* `High` : prix le plus haut de la journée
* `Low` : prix le plus bas de la journée
* `Close` : prix de clôture
* `Adj Close` : prix de clôture ajusté
* `Volume` : nombre d'actions échangées


In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
# Corriger le dataset avant de créer la cible.
# On vérifie les colonnes, on convertit les types, on enlève les lignes invalides
# et on trie les dates pour respecter l'ordre chronologique.
required_columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans le dataset: {missing_columns}")

df = df.copy()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

numeric_columns = ['Open', 'High', 'Low', 'Close', 'Volume']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Pour cette prédiction, on utilise Close comme prix cible.
# Adj Close est donc retirée pour éviter une colonne très proche de la cible.
if 'Adj Close' in df.columns:
    df = df.drop(columns=['Adj Close'])

df = (
    df.dropna(subset=required_columns)
      .drop_duplicates(subset=['Date'])
      .sort_values('Date')
      .reset_index(drop=True)
)


In [ ]:
# Utiliser Date comme index facilite la lecture des séries temporelles.
df.set_index('Date', inplace=True)


In [ ]:
# Vérification rapide après nettoyage.
# Les colonnes de prix et de volume doivent être numériques.
df.info()


In [ ]:

df.head()

In [ ]:
# Dernière sécurité: supprimer toute ligne encore incomplète après les conversions.
df = df.dropna()


In [ ]:
# Créer la cible: le prix de clôture du jour suivant.
# shift(-1) décale Close vers le haut; la dernière ligne n'a donc plus de cible.
df['Target'] = df['Close'].shift(-1)

# Cette ligne est essentielle: sans elle, y contient une valeur NaN.
df = df.dropna(subset=['Target'])

df.head()


In [ ]:
# Séparer les variables explicatives et la cible.
# Target ne doit jamais être utilisée comme entrée du modèle.
feature_columns = [column for column in df.columns if column != 'Target']
features = df[feature_columns]
target = df[['Target']]

print(f"Nombre de lignes utilisables: {len(df)}")
print(f"Colonnes utilisées comme features: {feature_columns}")


## **3. Préparer les données pour l'entraînement**

In [ ]:
# Pour une série temporelle, on évite train_test_split avec shuffle.
# Les 80% premières dates servent à l'entraînement; les 20% les plus récentes servent au test.
split_index = int(len(df) * 0.8)

X_train_raw = features.iloc[:split_index]
X_test_raw = features.iloc[split_index:]
y_train_raw = target.iloc[:split_index]
y_test_raw = target.iloc[split_index:]

# Les scalers sont ajustés uniquement sur l'entraînement.
# Cela évite d'utiliser indirectement des informations du test pendant l'apprentissage.
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train = feature_scaler.fit_transform(X_train_raw)
X_test = feature_scaler.transform(X_test_raw)
y_train = target_scaler.fit_transform(y_train_raw).ravel()
y_test = target_scaler.transform(y_test_raw).ravel()

print(f"Train: {X_train.shape[0]} lignes")
print(f"Test: {X_test.shape[0]} lignes")


In [ ]:
class StockPriceDataset(Dataset):
    def __init__(self, X, y):
        # PyTorch travaille avec des tensors.
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
# Création des datasets PyTorch.
# Chaque élément contient les features normalisées et la cible normalisée correspondante.
train_dataset = StockPriceDataset(X_train, y_train)
test_dataset = StockPriceDataset(X_test, y_test)


In [ ]:
# Création des DataLoaders.
# shuffle=True mélange les batchs d'entraînement, tandis que le test reste ordonné.
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")


## **4. Définir le modèle LSTM/GRU**

In [ ]:
class StockPriceModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, lstm_layers=1, gru_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers=lstm_layers,
            batch_first=True
        )
        self.gru = nn.GRU(
            hidden_size,
            hidden_size,
            num_layers=gru_layers,
            batch_first=True,
            dropout=dropout if gru_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # Le modèle attend une forme (batch, seq_len, input_size).
        # Ici, chaque exemple représente une journée: seq_len vaut donc 1.
        x = x.unsqueeze(1)
        out, _ = self.lstm(x)
        out, _ = self.gru(out)
        out = out[:, -1, :]
        out = self.dropout(out)
        return self.fc(out).squeeze(-1)


In [ ]:
input_size = X_train.shape[1]
model = StockPriceModel(input_size=input_size, hidden_size=64, lstm_layers=1, gru_layers=2, dropout=0.2)
print(model)

## **5. Train the Model**

In [ ]:
# Définition de la fonction de perte et de l'optimiseur.
# MSELoss convient à une prédiction numérique continue.
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# Entraînement du modèle.
# Les pertes sont gardées pour tracer les courbes ensuite.
num_epochs = 50
train_losses = []
val_losses = []


In [ ]:
for epoch in range(num_epochs):
    # Phase d'entraînement: le modèle ajuste ses poids.
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # Phase de validation: on mesure l'erreur sans modifier les poids.
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item()
    
    val_loss /= len(test_loader)
    val_losses.append(val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

print("Entraînement terminé!")


In [ ]:
# Visualiser les pertes d'entraînement et de validation.
# Une validation qui remonte fortement peut indiquer du surapprentissage.
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Model Training and Validation Loss')
plt.show()


## **6. Évaluer le modèle**

In [ ]:
# Évaluation sur l'ensemble de test et calcul du R².
# Les prédictions sont remises dans l'échelle originale du prix avant le calcul.
model.eval()
y_preds = []
y_trues = []

with torch.no_grad():
    for Xb, yb in test_loader:
        out = model(Xb)
        y_preds.append(out.cpu())
        y_trues.append(yb.cpu())

y_pred_scaled = torch.cat(y_preds).numpy().reshape(-1, 1)
y_true_scaled = torch.cat(y_trues).numpy().reshape(-1, 1)

y_pred = target_scaler.inverse_transform(y_pred_scaled).ravel()
y_true = target_scaler.inverse_transform(y_true_scaled).ravel()

ss_res = np.sum((y_true - y_pred) ** 2)
ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
r2 = 1 - ss_res / ss_tot if ss_tot != 0 else float('nan')
print(f"R² (test): {r2:.6f}")

# Conserver les scalers permet de transformer de nouvelles données plus tard.
saved_feature_scaler = feature_scaler
saved_target_scaler = target_scaler

# Tableau de comparaison entre valeurs réelles et valeurs prédites.
results_df = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred}, index=y_test_raw.index)
results_df.head()
